In [1]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical

# Load the serialized data
DATA_DIR = os.path.join("..", "data")
X = np.load(os.path.join(DATA_DIR, "X_features.npy"))
y = np.load(os.path.join(DATA_DIR, "y_labels.npy"))

print(f"Loaded X shape: {X.shape}")
print(f"Loaded y shape: {y.shape}")

RuntimeError: ('\nCould not import original test/binary location, import paths tried: [\'litert.python._pywrap_tensorflow_lite_metrics_wrapper\', \'tensorflow.lite.python._pywrap_tensorflow_lite_metrics_wrapper\', \'tensorflow._pywrap_tensorflow_lite_metrics_wrapper\', \'tensorflow.python._pywrap_tensorflow_lite_metrics_wrapper\']. \nPrevious exceptions: ["No module named \'litert\'", \'DLL load failed while importing _pywrap_tensorflow_lite_metrics_wrapper: The specified module could not be found.\', "No module named \'tensorflow._pywrap_tensorflow_lite_metrics_wrapper\'", "No module named \'tensorflow.python._pywrap_tensorflow_lite_metrics_wrapper\'"]', ModuleNotFoundError("No module named 'tensorflow.python._pywrap_tensorflow_lite_metrics_wrapper'"))

In [ ]:
# 1. Label Encoding (String -> Integer -> One-Hot)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)

# 2. Train/Test Split (80/20)
# stratify=y_encoded guarantees the exact same proportion of emotions in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

In [ ]:
# 3. Feature Scaling (Prevent Data Leakage by fitting strictly on X_train)
scaler = StandardScaler()

# Store original dimensions
train_shape = X_train.shape
test_shape = X_test.shape

# Scikit-learn scalers expect 2D arrays, so we flatten, scale, and reshape back to 3D
X_train_scaled = scaler.fit_transform(X_train.reshape(train_shape[0], -1)).reshape(train_shape)
X_test_scaled = scaler.transform(X_test.reshape(test_shape[0], -1)).reshape(test_shape)

# 4. Add the Channel Dimension (Mono Audio = 1 Channel)
X_train_final = np.expand_dims(X_train_scaled, axis=-1)
X_test_final = np.expand_dims(X_test_scaled, axis=-1)

print("\n--- DEEP LEARNING TENSORS READY ---")
print(f"Final X_train shape for CNN: {X_train_final.shape}")
print(f"Final X_test shape for CNN: {X_test_final.shape}")